# Türkçe Verifiable QA Pipeline — Colab Runner

Bu defter [pqa](https://github.com/cxrbon16/pqa) reposundaki pipeline'ı Colab'da uçtan uca çalıştırır.

**Önce:** `Çalışma zamanı > Çalışma zamanı türünü değiştir` menüsünden **G4** (NVIDIA RTX PRO 6000,
~96GB VRAM) seç. Bu defterdeki model seçimi G4'ün bol VRAM'ını kullanacak şekilde yapıldı —
T4/L4 gibi daha küçük bir GPU'daysan aşağıdaki "Daha küçük GPU" notuna bak. Sonra hücreleri sırayla
çalıştır.

In [ ]:
!nvidia-smi

## 1. Repoyu klonla ve bağımlılıkları kur

Colab'ın kendi Python ortamı zaten oturuma özel (izole) olduğu için ayrı bir venv gerekmiyor.

In [ ]:
!git clone https://github.com/cxrbon16/pqa.git
%cd pqa
!pip install -q -r requirements.txt

## 2. Ollama'yı kur, başlat, modelleri indir

`config.yaml`'daki model seçimi bilinçli olarak çeşitlendirildi:

| Rol | Model | Neden |
|---|---|---|
| Üretici (soru üretimi) | `gemma4:31b-q8_0` | Gemma 4'ün en güçlü dense boyutu, 8-bit — Türkçe dahil 140+ dilde güçlü, üretim en kalite-kritik adım olduğu için 4-bit yerine 8-bit |
| Çözücü 1 | `qwen3:32b` | Farklı aile (Alibaba), güçlü çok dilli akıl yürütme |
| Çözücü 2 | `llama4:scout` | Farklı aile (Meta), 109B toplam/17B aktif MoE |
| Çözücü 3 | `gemma4:26b` | Google MoE (3.8B aktif) — üreticiyle aynı aile ama farklı boyut/mimari |
| Çözücü 4 | `gemma4:e4b` | Kasıtlı küçük/zayıf model — hepsi çözerse soru "trivial" sayılıp elenir; zorluk bandı (`band.py`) ancak çözücüler arasında güç farkı (spread) varsa anlamlı olur |

**Toplam indirme ~110GB** (en büyük pay `llama4:scout`, ~65GB) — bant genişliğine göre 30-60+ dakika
sürebilir. Disk kotanı önce kontrol et; yetmiyorsa aşağıdaki "Daha küçük GPU" notundaki hafif
alternatifi kullan.

In [ ]:
!df -h /

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time

# G4'ün ~96GB VRAM'i birden fazla modeli aynı anda bellekte tutmaya izin verir (özellikle
# llama4:scout hariç diğerleri ~50GB'de rahat sığar) — bu da solve aşamasındaki paralelliği
# gerçek anlamda hızlandırır. Ollama gerektiğinde otomatik olarak modelleri yer değiştirir.
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "3"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print("ollama serve pid:", ollama_proc.pid)

In [ ]:
!ollama pull gemma4:31b-q8_0
!ollama pull qwen3:32b
!ollama pull llama4:scout
!ollama pull gemma4:26b
!ollama pull gemma4:e4b

In [ ]:
# Sunucu ayakta mı ve modeller indi mi kontrol et
!curl -s http://localhost:11434/api/tags | python3 -m json.tool

**Daha küçük GPU (T4/L4) kullanıyorsan:** Yukarıdaki model seti G4'ün ~96GB VRAM'ı için seçildi ve
T4/L4'te (16-24GB) sığmaz. `config.yaml`'da şu hafif sete geç:

```yaml
generation:
  model:
    model: gemma4:e4b
solving:
  solvers:
    - {name: qwen3-8b,   model: qwen3:8b,   base_url: http://localhost:11434/v1}
    - {name: llama4-e4b, model: gemma4:e4b, base_url: http://localhost:11434/v1}
    - {name: gemma4-e2b, model: gemma4:e2b, base_url: http://localhost:11434/v1}
```

ve buna göre `ollama pull` komutlarını güncelle.

## 3. Duman testi (smoke test): küçük bir örnekle uçtan uca çalıştır

`--limit N` her aşamanın girdisini ilk N kayıtla sınırlar. Önce küçük çalıştırıp pipeline'ın
uçtan uca sorunsuz aktığını doğrula.

In [ ]:
!python -m vqa all --limit 5

In [ ]:
import json

for line in open("data/06_final.jsonl", encoding="utf-8"):
    rec = json.loads(line)
    print(f"[{rec['difficulty']:>6} | solve_rate={rec['solve_rate']}] {rec['question']} -> {rec['answer']}")

## 4. Google Drive'a bağla (oturum kesilirse veri kaybolmasın)

Colab oturumları zaman aşımına uğrayabilir/kopabilir; her diskteki her şey oturumla birlikte
silinir. `data_dir`'i Drive'a taşımak, uzun bir çalıştırmayı kaldığı yerden sürdürmeni sağlar —
her aşama kendi JSONL'ini yazdığı için pipeline zaten yeniden başlatılabilir.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/pqa_data
!sed -i 's|^data_dir: .*|data_dir: /content/drive/MyDrive/pqa_data|' config.yaml
!grep data_dir config.yaml

## 5. Tam çalıştırma

`--limit` verme; pasaj/soru sayısını `config.yaml` → `source.hf_dataset.n_articles` ile ayarla
(pilot hedefi ~1k soru ≈ 600-1000 pasaj → `n_articles` ~250-350 civarı, `passages.max_passages_per_article`
değerine bağlı olarak değişir — küçük bir aralıkla deneyip oranı gözlemlemen daha güvenli).

Bağlantı koparsa: bu hücreyi (veya tek tek aşama komutlarını) tekrar çalıştırman yeterli,
tamamlanmış aşamalar `data_dir` altındaki JSONL'lerden anlaşılır ve `vqa` her aşamayı
girdisi hazır olduğu sürece bağımsız çalıştırır.

In [ ]:
!python -m vqa all

## 6. Hugging Face'e yayınla

`config.yaml` → `publish.repo_id` alanını kendi kullanıcı adınla güncelle, sonra giriş yap.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!python -m vqa publish